# Coop Case — Q4.5: Is Coop's Discount Spend Going to the Right Customers?

**Question:** Different lens than Q4.2 (which asked whether discounts drive *frequency*). This asks:
**who is actually receiving Coop's discount SEK** -- is it concentrated on already-loyal, high-value
customers (arguably wasted, since they'd likely have bought anyway), or spread toward lower-value/
marginal customers (a more efficient use as an acquisition or retention lever)?


## 1. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)


In [ ]:
DATA_PATH = "2months_v2/rl_2months.csv"

dtypes = {
    "receiptKey": "int64",
    "hourOfDay": "int8",
    "minuteOfHour": "int8",
    "quantity": "float32",
    "lineItemAmount": "float32",
    "lineItemAmountExclVat": "float32",
    "discountAmountExclVat": "float32",
    "lineItemCostExclVat": "float32",
    "CoopOnlineYN": "category",
    "store": "category",
    "customerId": "Int64",
    "householdId": "Int64",
    "MOSAICGroup": "category",
    "MOSAICGroupDescription": "category",
    "MOSAICType": "category",
    "MOSAICTypeDescription": "category",
    "DominantBuyingPowerClass": "category",
    "ItemID": "int64",
    "ItemSubSegmentName": "category",
    "ItemSubSegmentID": "Int64",
    "ItemSegmentName": "category",
    "ItemSegmentID": "Int64",
    "ItemSubCategoryName": "category",
    "ItemSubCategoryID": "Int64",
    "ItemCategoryName": "category",
    "ItemCategoryID": "Int64",
    "ItemCategoryTeamName": "category",
    "ItemCategoryTeamID": "Int64",
    "ItemCategoryGroupName": "category",
    "ItemCategoryGroupID": "Int64",
    "ItemCategoryAreaName": "category",
    "ItemCategoryAreaID": "Int64",
    "Brand": "category",
    # Stored as floats in the source file (0.0 / 1.0), not clean ints -- pandas
    # won't safely downcast float64 -> int8 during read_csv, so keep as float32.
    "eko": "float32",
    "organic": "float32",
    "krav": "float32",
    "fair_trade": "float32",
    "msc": "float32",
    "no_lactose": "float32",
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=["DayDate"],
)
df["profit"] = df["lineItemAmountExclVat"] - df["lineItemCostExclVat"]

print(df.shape)
df.head()

## 2. Household-level discount and value totals

In [ ]:
household = df.groupby("householdId", observed=True).agg(
    revenue=("lineItemAmountExclVat", "sum"),
    profit=("profit", "sum"),
    discount=("discountAmountExclVat", "sum"),
).reset_index()
household["discount_sek"] = -household["discount"]  # positive SEK value of discount received

total_discount = household["discount_sek"].sum()
total_profit = household["profit"].sum()
total_revenue = household["revenue"].sum()

print(f"Total discount given (2 months, both stores): {total_discount:,.0f} SEK")
print(f"Total profit: {total_profit:,.0f} SEK")
print(f"Discount as % of revenue: {total_discount / total_revenue:.2%}")
print(f"Discount as % of profit (margin given up, relative to margin kept): {total_discount / total_profit:.2%}")


## 3. Do the highest-value households get a disproportionate share of discounts?

Rank households by profit, then compare each group's share of total *discount* SEK against its
share of total *profit* -- the same concentration lens used for the Q3 Lorenz curve, applied to
discounts instead of raw value.


In [ ]:
household_sorted = household.sort_values("profit", ascending=False).reset_index(drop=True)
household_sorted["cum_profit_pct"] = household_sorted["profit"].cumsum() / household_sorted["profit"].sum() * 100
household_sorted["cum_discount_pct"] = household_sorted["discount_sek"].cumsum() / household_sorted["discount_sek"].sum() * 100
household_sorted["cum_hh_pct"] = (household_sorted.index + 1) / len(household_sorted) * 100

for pct in [10, 20, 50]:
    row = household_sorted[household_sorted["cum_hh_pct"] >= pct].iloc[0]
    print(f"Top {pct}% of households (by profit) receive {row['cum_discount_pct']:.1f}% of total discount SEK "
          f"(vs. {row['cum_profit_pct']:.1f}% of total profit)")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(household_sorted["cum_hh_pct"], household_sorted["cum_profit_pct"], color="#22B573", linewidth=2.5, label="Cumulative % of profit")
ax.plot(household_sorted["cum_hh_pct"], household_sorted["cum_discount_pct"], color="#E8734A", linewidth=2.5, label="Cumulative % of discount SEK")
ax.plot([0, 100], [0, 100], linestyle="--", color="#8FA097", label="Perfect equality")
ax.set_xlabel("Cumulative % of households (ranked by profit, highest first)")
ax.set_ylabel("Cumulative %")
ax.set_title("Profit concentration vs. discount concentration\nDiscounts are LESS concentrated than profit -- spread more toward lower-value households")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Discount intensity by value tier

Split households into value quartiles (by profit), and compute **discount intensity** = total
discount SEK / total revenue **within each tier** (aggregated, not averaged per household -- avoids
blow-up from near-zero-revenue households).


In [ ]:
household["value_tier"] = pd.qcut(household["profit"].rank(method="first"), 4,
                                    labels=["Q1 (lowest value)", "Q2", "Q3", "Q4 (highest value)"])

tier_summary = household.groupby("value_tier", observed=True).agg(
    n_households=("householdId", "count"),
    avg_profit=("profit", "mean"),
    total_revenue=("revenue", "sum"),
    total_discount=("discount_sek", "sum"),
)
tier_summary["discount_intensity_pct"] = tier_summary["total_discount"] / tier_summary["total_revenue"] * 100
tier_summary.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tier_summary["discount_intensity_pct"].plot(kind="bar", ax=ax, color="#22B573")
ax.set_title("Discount intensity (discount SEK as % of revenue) by household value tier")
ax.set_ylabel("Discount intensity (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


## 5. Takeaways

- **Discounts are LESS concentrated than profit -- not more.** Top 10% of households (by profit)
  earn Coop **47.9%** of total profit but receive only **35.0%** of total discount SEK. Top 20%
  drive **69.0%** of profit but get **53.5%** of discounts. Top 50% drive **94.1%** of profit but
  get **79.8%** of discounts. At every cut, high-value households receive a *smaller* share of
  discount spend than their share of profit -- the opposite of "wasting discounts on people who'd
  buy anyway."

- **Discount intensity has a clean, monotonic gradient by value tier**: lowest-value households
  (Q1) have **24.3%** of their revenue discounted, vs. **11.8%** (Q2), **9.3%** (Q3), and just
  **7.2%** for the highest-value households (Q4). Lower-value/more marginal customers are getting
  proportionally much deeper discounting than Coop's best customers.

- **Overall discount scale**: Coop gave away **2.50M SEK** in discounts over this 2-month sample --
  **8.5% of revenue**, or **30.6% of gross margin** (i.e. discounts given up are worth about
  30 cents of every profit-SEK Coop actually kept). That's a substantial share of margin, worth
  knowing in absolute terms regardless of who it's going to.

- **How to read this**: on the surface this looks efficient -- discount spend is skewed toward
  lower-value, more marginal customers rather than "wasted" on already-loyal top spenders. But it
  cuts both ways: it could mean discounts are doing real acquisition/retention work on the customers
  who need the nudge (good), *or* it could mean Coop's highest-value customers are being under-invested
  in precisely because they don't need convincing -- a risk if a competitor targets them instead.
  This dataset can't distinguish those two stories on its own.

- **What's missing to fully answer this**: causal evidence. We know *where* discount SEK flows, not
  whether it's *causing* incremental purchases from low-value households or whether those households
  would have bought a similar (smaller) basket anyway. Same limitation as Q4.2 — the fix is a
  controlled test: hold back discounts from a matched sample of low-value households and see if
  their spend actually drops, versus a group that keeps receiving them.
